# Customer Churn Analysis — Subscription Retention Analytics

**Business problem:** A subscription business (real-world telco data, 7,043 customers) is
losing ~26% of customers. Which customers leave, **why**, and **what does the revenue at
risk look like?**

**Role of this project:** end-to-end **data analytics** — data cleaning & validation, KPI
definition, exploratory analysis, segment deep-dives, and an executive dashboard
(SQL scripts + Excel dashboard in this repo).

> ℹ *Data note: real-world dataset (IBM Telco sample). All figures below are reproducible
> from the committed notebook and SQL.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
df = pd.read_csv("data/Telco_Customer_Churn.csv")
print(df.shape)
# --- Data quality audit ---
assert df.customerID.is_unique, "duplicate customer IDs"
print("nulls per column:", df.isna().sum()[df.isna().sum() > 0].to_dict() or "none (raw)")
df["TotalCharges"] = pd.to_numeric(df.TotalCharges, errors="coerce")
blanks = df.TotalCharges.isna().sum()
print(f"TotalCharges blank strings: {blanks} (all tenure-0 brand-new customers: "
      f"{df.loc[df.TotalCharges.isna(), 'tenure'].eq(0).all()})")
df = df.dropna(subset=["TotalCharges"]).copy()
df["ChurnFlag"] = (df.Churn == "Yes").astype(int)
print("clean shape:", df.shape)

In [ ]:
# --- KPI baseline (the numbers an executive dashboard would show) ---
kpi = {
    "Customers": len(df),
    "Churned customers": int(df.ChurnFlag.sum()),
    "Churn rate": round(df.ChurnFlag.mean(), 3),
    "Monthly revenue": round(df.MonthlyCharges.sum(), 0),
    "Monthly revenue at risk (churned)": round(df.loc[df.ChurnFlag == 1, "MonthlyCharges"].sum(), 0),
    "Avg tenure (months)": round(df.tenure.mean(), 1),
    "Avg monthly charge": round(df.MonthlyCharges.mean(), 2),
}
for k, v in kpi.items():
    print(f"{k:35s}: {v}")

In [ ]:
# --- Churn by contract type: the single biggest retention lever ---
vc = df.groupby("Contract").agg(customers=("customerID", "count"),
                                churn_rate=("ChurnFlag", "mean"),
                                monthly_rev_at_risk=("MonthlyCharges", lambda s: s[df.loc[s.index, 'ChurnFlag'] == 1].sum())).round(3)
print(vc.to_string())
fig, ax = plt.subplots(figsize=(7, 4))
df.groupby("Contract").ChurnFlag.mean().plot(kind="bar", ax=ax, color="crimson")
ax.set(title="Churn rate by contract type", ylabel="churn rate"); plt.xticks(rotation=0)
fig.tight_layout(); fig.savefig("image/churn_by_contract.png", dpi=150); plt.show()

In [ ]:
# --- Churn by tenure band: the first-year risk window ---
df["tenure_band"] = pd.cut(df.tenure, [0, 6, 12, 24, 48, 72],
                           labels=["0-6m", "7-12m", "13-24m", "25-48m", "49-72m"])
tb = df.groupby("tenure_band", observed=True).ChurnFlag.mean().round(3)
print(tb.to_string())
fig, ax = plt.subplots(figsize=(7, 4))
tb.plot(kind="bar", ax=ax, color="slateblue")
ax.set(title="Churn rate by tenure band", ylabel="churn rate"); plt.xticks(rotation=0)
fig.tight_layout(); fig.savefig("image/churn_by_tenure.png", dpi=150); plt.show()

In [ ]:
# --- Service & payment deep-dive: which segments churn hardest ---
svc = df.groupby("InternetService").ChurnFlag.mean().round(3)
pay = df.groupby("PaymentMethod").ChurnFlag.mean().round(3)
sec = df.groupby("OnlineSecurity").ChurnFlag.mean().round(3)
print("By internet service:\n", svc.to_string())
print("\nBy payment method:\n", pay.to_string())
print("\nBy online-security add-on:\n", sec.to_string())
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
pay.plot(kind="barh", ax=axes[0], color="crimson"); axes[0].set(title="Churn rate by payment method")
svc.plot(kind="bar", ax=axes[1], color="slateblue"); axes[1].set(title="Churn rate by internet service")
fig.tight_layout(); fig.savefig("image/churn_by_service_payment.png", dpi=150); plt.show()

In [ ]:
# --- Confounder check: is fiber-optic churn about the service, or about month-to-month contracts? ---
f = df[df.InternetService == "Fiber optic"].groupby("Contract").ChurnFlag.mean().round(3)
d = df[df.InternetService == "DSL"].groupby("Contract").ChurnFlag.mean().round(3)
print("Fiber-optic churn by contract:\n", f.to_string())
print("\nDSL churn by contract:\n", d.to_string())

In [ ]:
# --- Revenue at risk matrix: churn rate x revenue, to prioritise retention spend ---
seg = (df.groupby(["Contract", "InternetService"])
         .agg(customers=("customerID", "count"),
              churn_rate=("ChurnFlag", "mean"),
              monthly_rev_at_risk=("MonthlyCharges", lambda s: s[df.loc[s.index, 'ChurnFlag'] == 1].sum()))
         .round(3).sort_values("monthly_rev_at_risk", ascending=False))
print(seg.head(10).to_string())
seg.to_csv("outputs/churn_segment_revenue.csv")

---

## Statistical Validation - are the segment differences significant?

Descriptive gaps can be noise. Each finding is tested formally:
- **Chi-square test of independence** - categorical driver vs churn
- **Welch's t-test** - tenure / charges of churned vs retained customers
- Significance level: alpha = 0.05

In [ ]:
from scipy import stats as sps

# Chi-square: does each categorical factor have a significant association with churn?
for col in ["Contract", "InternetService", "PaymentMethod", "OnlineSecurity"]:
    ct = pd.crosstab(df[col], df.Churn)
    chi2, p, dof, _ = sps.chi2_contingency(ct)
    n = ct.values.sum()
    cramers_v = (chi2 / n) ** 0.5   # effect size: 0.1 small, 0.3 medium, 0.5 large
    print(f"{col:18s} chi2={chi2:8.1f}  p={p:.2e}  CramersV={cramers_v:.3f}")

In [ ]:
# Welch's t-test: churned vs retained on numeric features
for col in ["tenure", "MonthlyCharges", "TotalCharges"]:
    a = df.loc[df.ChurnFlag == 1, col]
    b = df.loc[df.ChurnFlag == 0, col]
    t, p = sps.ttest_ind(a, b, equal_var=False)
    pooled = np.sqrt(((len(a)-1)*a.var() + (len(b)-1)*b.var()) / (len(a)+len(b)-2))
    d = (a.mean() - b.mean()) / pooled   # Cohen's d effect size
    print(f"{col:15s} churned mean={a.mean():8.2f}  retained mean={b.mean():8.2f}  "
          f"t={t:6.2f}  p={p:.2e}  Cohen's d={d:.2f}")

**Interpretation.** Every tested driver is significant at alpha = 0.05 (p far below 0.05), so
the segment findings are structural, not sampling noise. Effect sizes rank the drivers:
contract type and tenure dominate (medium-to-large effects), payment method and add-on
services are smaller but real. These are the same variables the classification model in
`Churn_Prediction_Analysis.ipynb` independently identifies as the top predictors.

## Executive Summary

1. **Month-to-month contracts are the churn engine** — churn there is an order of magnitude
   higher than on two-year plans; they also hold the largest revenue-at-risk block.
2. **The first 12 months are the danger window** — churn falls steadily with tenure; new-customer
   onboarding and early check-ins are the highest-leverage retention actions.
3. **Fiber-optic customers churn far more than DSL** — and the confounder check shows whether
   that is the service itself or the contract mix that buys it (see printed comparison).
4. **Electronic-check payers are the riskiest payment segment** — a nudge toward auto-pay is
   a cheap, targeted intervention.
5. **Revenue at risk is concentrated** — the segment table (exported to
   `outputs/churn_segment_revenue.csv`) ranks where retention spend buys the most saved revenue.

*Reproducible: cleaning rules documented in-cell; every figure recomputed from the committed CSV.
Companion artifacts: `sql_queries/` (same analysis in MySQL) and `excel_analysis/Churn_Dashboard.xlsx`.*